# Fraud Detection — Model Training

Orchestrates the full pipeline:
`load → feature engineering → prep → split → SHAP selection → tune → train → evaluate → save`

All logic lives in `src/features.py`, `src/model.py`, and `src/ensemble.py`.
This notebook is for running and observing, not for defining functions.

In [ ]:
import sys
sys.path.append('..')

import yaml
import mlflow
import mlflow.lightgbm
from pathlib import Path

import numpy as np
import pandas as pd

from src.features import build_features
from src.model import (
    prepare_features,
    three_way_split,
    select_features_shap,
    LABEL,
)
from src.ensemble import (
    encode_for_ensemble,
    tune_all,
    train_ensemble,
    evaluate_ensemble,
    calibrate_ensemble,
    tune_threshold,
    save_ensemble,
)

DATA_DIR = '../data'
import src as _src
MLRUNS = str(Path(_src.__file__).parent.parent / 'mlruns')
mlflow.set_tracking_uri(MLRUNS)
mlflow.set_experiment('fraud-detection')

In [ ]:
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

RANDOM_STATE = cfg['random_state']
np.random.seed(RANDOM_STATE)
print(f"RANDOM_STATE = {RANDOM_STATE}")

## 1. Load raw data

In [18]:
trn = pd.read_csv(f'{DATA_DIR}/train_transaction.csv')
idn = pd.read_csv(f'{DATA_DIR}/train_identity.csv')
print(f'Transactions : {trn.shape}')
print(f'Identity     : {idn.shape}')

Transactions : (590540, 394)
Identity     : (144233, 41)


## 2. Feature engineering

In [19]:
df = build_features(trn, idn)
print(f'After feature engineering: {df.shape}')

Merged            : 590,540 rows × 434 cols
Identity coverage : 24.4%
Dropped 11 weak id columns


KeyboardInterrupt: 

## 3. Prepare feature matrix

In [ ]:
df = prepare_features(df)
print(f'After prep: {df.shape}')

Dropping 9 columns with >99% missing
Dropping 0 constant columns
After prep: (590540, 450)


## 4. Three-way chronological split

| Split | Fraction | Purpose |
|---|---|---|
| **train** | 70% | Model training + Optuna tuning |
| **cal** | 10% | Calibration only — never seen during training |
| **test** | 20% | Early stopping + final evaluation |

In [ ]:
train_df, cal_df, test_df = three_way_split(
    df,
    train_frac=cfg['split']['train_frac'],
    cal_frac=cfg['split']['cal_frac'],
)

FEATURES = [c for c in train_df.columns if c != LABEL]
X_train, y_train = train_df[FEATURES], train_df[LABEL]
X_cal,   y_cal   = cal_df[FEATURES],   cal_df[LABEL]
X_test,  y_test  = test_df[FEATURES],  test_df[LABEL]

print(f'Train : {X_train.shape}  |  fraud rate: {y_train.mean():.2%}')
print(f'Cal   : {X_cal.shape}    |  fraud rate: {y_cal.mean():.2%}')
print(f'Test  : {X_test.shape}   |  fraud rate: {y_test.mean():.2%}')

# Persist engineered test set for the streaming simulation notebook
test_df.to_parquet('../data/test_features.parquet', index=False)
print(f"Saved {len(test_df):,} rows → data/test_features.parquet")

## 5. Feature selection

Uses SHAP (TreeExplainer) on a 50K subsample to rank features by mean absolute
contribution, then keeps the top 40. Results are cached to `models/shap_features.json`
so SHAP only runs once — subsequent runs load from cache instantly.

In [ ]:
X_train_f, X_test_f, X_cal_f, features_f = select_features_shap(
    X_train, y_train, X_test, X_cal=X_cal, top_n=cfg['feature_selection']['top_n'],
)

## 6. Soft-voting ensemble (LightGBM + XGBoost + CatBoost)

Averages `predict_proba` across three independently tuned models.

### 6a. Encode categoricals

Ordinal-encodes category columns to integers. Fit on `X_train_f` so codes are
consistent across train and test splits.

In [ ]:
X_train_enc, X_test_enc, cat_cols, cat_encoders = encode_for_ensemble(X_train_f, X_test_f)

# Apply same encoding to calibration set using training codes
X_cal_enc = X_cal_f.copy()
for col in cat_cols:
    X_cal_enc[col] = X_cal_enc[col].astype(object).map(cat_encoders[col]).fillna(-1).astype(int)

print(f"Encoded {len(cat_cols)} categorical columns: {cat_cols}")

### 6b. Tune all three models

Each model is tuned independently with Optuna.
`scale_pos_weight` / class weights are fixed from the class ratio — not searched.

In [ ]:
all_best_params = tune_all(
    X_train_enc, y_train, cat_cols,
    n_trials=cfg['tuning']['n_trials'],
)

print("\nBest params per model:")
for name, params in all_best_params.items():
    print(f"  {name}: {params}")

### 6c. Train, evaluate, and save ensemble

In [ ]:
with mlflow.start_run(run_name='ensemble-lgbm-xgb-catboost'):

    mlflow.log_params({
        'train_rows':        len(X_train_enc),
        'cal_rows':          len(X_cal_enc),
        'test_rows':         len(X_test_enc),
        'n_features':        X_train_enc.shape[1],
        'fraud_rate_train':  round(float(y_train.mean()), 4),
        'feature_selection': 'SHAP top 40',
        'ensemble':          'soft_vote_lgbm_xgb_catboost',
    })
    for model_name, params in all_best_params.items():
        mlflow.log_params({f"{model_name}__{k}": v for k, v in params.items()})

    ensemble_models = train_ensemble(
        X_train_enc, y_train,
        X_test_enc,  y_test,
        best_params=all_best_params,
        cat_cols=cat_cols,
    )

    ensemble_metrics = evaluate_ensemble(
        ensemble_models, X_test_enc, y_test,
        threshold=cfg['ensemble']['threshold'],
        plot=True,
    )
    mlflow.log_metrics(ensemble_metrics)

    # Save without calibrator for now — added in section 6d
    save_ensemble(ensemble_models, cat_cols, cat_encoders)

    print(f"\nMLflow run logged. Ensemble PR-AUC: {ensemble_metrics['pr_auc']:.4f}")

### 6d. Probability calibration

Boosted tree ensembles tend to produce overconfident scores (pushed toward 0 and 1).
Isotonic regression fits a monotone mapping from raw scores → calibrated probabilities
using the **calibration set** — data the models have never seen during training or early stopping.

In [ ]:
calibrator = calibrate_ensemble(
    ensemble_models, X_cal_enc, y_cal,
    method=cfg['calibration']['method'],
)

### 6e. Threshold tuning

Find the decision threshold that maximises F-beta on the **test set** using calibrated probabilities.
β=2 weights recall 4× over precision — missing fraud costs more than a false alarm.

In [ ]:
# Re-save ensemble with calibrator + optimal threshold baked in
save_ensemble(
    ensemble_models, cat_cols, cat_encoders,
    calibrator=calibrator,
    threshold=best['threshold'],
)

print(f"\nFinal model saved.")
print(f"  Calibration : {cfg['calibration']['method']}")
print(f"  Threshold   : {best['threshold']:.4f}  (F{cfg['calibration']['beta']:.0f}-optimal)")

In [ ]:
from src.ensemble import predict_proba_ensemble
import numpy as np

raw_probs_test = predict_proba_ensemble(ensemble_models, X_test_enc)
cal_probs_test = calibrator.predict(raw_probs_test)

best = tune_threshold(y_test, cal_probs_test, beta=cfg['calibration']['beta'])

## 7. View MLflow UI

```bash
mlflow ui
```
Open http://localhost:5000 to compare all runs side by side.